# 一括実行

メインフォルダを 1 回選ぶだけで、特徴量抽出の 3 つのノートブック（最高値版・相対値方式・加速度方式）を順に実行し、結果をまとめて保存します。
各ノートブックのコードをそのまま読み込んで実行するので、それぞれのセル2（設定）やセル6（グラフ設定）を変えれば、ここにも反映されます。

**使い方**
1. カーネルに `.venv`（resin-voltage-feature-extraction 3.12.12）を選ぶ
2. 必要ならセル1の設定を変える
3. セル1 → セル2 → セル3（メインフォルダを選ぶ）→ セル4 の順に実行する

**出力**：`全コード一括実行/実行結果/<メインフォルダ名>_<日時>/`
- `最高値版/`：最高値version / 電圧特徴量(最高値他)抽出
- `相対値方式/`：変曲点version / 電圧特徴量抽出(相対値方式で変曲点抽出)
- `加速度方式/`：変曲点version / 電圧特徴量抽出(加速度方式で変曲点抽出)
- `圧縮データ/`：最高値version / データ圧縮コード（`RUN_COMPRESSION = True` のときだけ）
- `実行ログ.txt`：画面に表示された内容の記録

**注意**
- 時間がかかります（1 ファイルあたり、3 つのノートブックの合計で約 10 秒。300 ファイルなら 1 時間弱）。実行中はパソコンをスリープさせないでください。
- 各ノートブックのコードセルの順番（セル1〜6）を前提にしています。セルを追加したり並べ替えたりしたときは、このノートブックも見直してください。
- `全コード一括実行/実行結果/` は Git の対象外です（`.gitignore`）。
- このノートブックは `全コード一括実行/` に置いたまま使ってください。ほかのノートブックの場所（プロジェクト直下）は自動で探します。

In [5]:
# セル1 --- 設定 ---

# 加工対象ファイルが入っているメインフォルダ
# None のときは、セル3でダイアログを開いて選びます。パスを直接書いてもかまいません（例: r'D:\実験データ\2026_09'）
MAIN_FOLDER = None

# 結果の保存先（「全コード一括実行」フォルダからの相対パス。絶対パスを書いてもかまいません）
OUTPUT_ROOT = '実行結果'

# True にすると、データ圧縮コードも全ファイルに実行します（1 ファイルごとに圧縮 CSV とグラフ PNG を出力）
RUN_COMPRESSION = False

print("セル1: 設定の読み込みが完了しました。")

セル1: 設定の読み込みが完了しました。


In [6]:
# セル2 --- 関数の定義 ---

import os
import sys
import glob
import json
import time
import traceback
import contextlib
from datetime import datetime
import tkinter as tk
from tkinter import filedialog
import pandas as pd

def find_project_root(start):
    """start から上の階層へさかのぼり、最高値version と 変曲点version の両方があるフォルダ（プロジェクト直下）を返す"""
    folder = os.path.abspath(start)
    while True:
        if all(os.path.isdir(os.path.join(folder, name)) for name in ('最高値version', '変曲点version')):
            return folder
        parent = os.path.dirname(folder)
        if parent == folder:
            return None
        folder = parent


# VS Code は、このノートブックがあるフォルダ（全コード一括実行）を作業フォルダにして実行する
PROJECT_ROOT = find_project_root(os.getcwd())
BATCH_DIR = os.path.join(PROJECT_ROOT, '全コード一括実行') if PROJECT_ROOT else None

# 実行する特徴量抽出ノートブック（出力サブフォルダ名, プロジェクト直下からのパス）
FEATURE_NOTEBOOKS = [
    ('最高値版',         '最高値version/コードフォルダ/電圧特徴量(最高値他)抽出.ipynb'),
    ('相対値方式',       '変曲点version/コードフォルダ/電圧特徴量抽出(相対値方式で変曲点抽出).ipynb'),
    ('加速度方式',       '変曲点version/コードフォルダ/電圧特徴量抽出(加速度方式で変曲点抽出).ipynb'),
]
COMPRESSION_NOTEBOOK = ('圧縮データ', '最高値version/コードフォルダ/データ圧縮コード_1ファイルのみ.ipynb')


def load_code_cells(nb_path):
    """ノートブックの空でないコードセルを、上から順に返す"""
    with open(os.path.join(PROJECT_ROOT, nb_path), encoding='utf-8') as f:
        cells = json.load(f)['cells']
    sources = [''.join(c['source']) for c in cells if c['cell_type'] == 'code']
    return [s for s in sources if s.strip()]


@contextlib.contextmanager
def dialog_returns(path):
    """各ノートブックのフォルダ選択ダイアログを開かずに、path を返すようにする（終わったら元に戻す）"""
    class _DummyTk:
        def withdraw(self): pass
        def attributes(self, *args): pass
    original_tk, original_ask = tk.Tk, filedialog.askdirectory
    tk.Tk = _DummyTk
    filedialog.askdirectory = lambda **kwargs: path
    try:
        yield
    finally:
        tk.Tk, filedialog.askdirectory = original_tk, original_ask


class Tee:
    """画面への表示を、ログファイルにも書き込む"""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for s in self.streams:
            s.write(text)
        return len(text)
    def flush(self):
        for s in self.streams:
            s.flush()


def run_feature_notebook(nb_path, main_folder, out_dir):
    """特徴量抽出ノートブックのセル1〜6を実行し、(検出した CSV の一覧, 特徴量一覧 CSV のパス) を返す"""
    cells = load_code_cells(nb_path)
    ns = {'__name__': '__main__'}
    for src in cells[:3]:                 # セル1〜3：ライブラリ・設定・関数
        exec(src, ns)
    with dialog_returns(main_folder):     # セル4：メインフォルダから CSV を探す
        exec(cells[3], ns)
    if not ns.get('csv_files'):
        raise RuntimeError('処理できる CSV が見つかりませんでした。')
    ns['folder_path'] = out_dir           # ここから先の出力先だけを差し替える
    exec(cells[4], ns)                    # セル5：特徴量一覧 CSV
    exec(cells[5], ns)                    # セル6：グラフ一覧 Excel
    if not ns.get('output_summary_path'):
        raise RuntimeError('特徴量一覧 CSV が作られませんでした。')
    return ns['csv_files'], ns['output_summary_path']


def run_compression(nb_path, main_folder, csv_files, out_dir):
    """データ圧縮コードを全ファイルに実行する（メインフォルダと同じサブフォルダ構成で out_dir に保存）。失敗したファイルの一覧を返す"""
    import matplotlib.pyplot as plt
    cells = load_code_cells(nb_path)
    target = 'output_dir = os.path.dirname(filepath)'
    if target not in cells[4]:
        raise RuntimeError('データ圧縮コードのセル5に、出力先を決める行が見つかりません。')
    cell5 = cells[4].replace(target, 'output_dir = COMPRESSION_OUT_DIR')
    ns = {'__name__': '__main__'}
    for src in cells[:2]:                 # セル1・2：ライブラリ・設定（セル3のファイル選択は使わない）
        exec(src, ns)
    failed = []
    original_show = plt.show
    plt.show = lambda *args, **kwargs: None
    try:
        for filepath in sorted(csv_files):
            rel = os.path.relpath(filepath, main_folder)
            print(f"\n--- 圧縮: {rel} ---")
            ns['COMPRESSION_OUT_DIR'] = os.path.normpath(os.path.join(out_dir, os.path.dirname(rel)))
            os.makedirs(ns['COMPRESSION_OUT_DIR'], exist_ok=True)
            ns['filepath'] = filepath
            for name in ('df_clipped', 'df_compressed'):
                ns.pop(name, None)        # 前のファイルの結果が残らないようにする
            try:
                exec(cells[3], ns)        # セル4：平滑化・切り出し・間引き
                exec(cell5, ns)           # セル5：圧縮 CSV とグラフ PNG の出力
            except Exception:
                traceback.print_exc(file=sys.stdout)
                failed.append(rel)
            finally:
                plt.close('all')
    finally:
        plt.show = original_show
    return failed


if PROJECT_ROOT is None:
    print("エラー: プロジェクトのフォルダ（最高値version と 変曲点version があるフォルダ）が見つかりません。")
    print("       このノートブックは、プロジェクト内の「全コード一括実行」フォルダに置いて実行してください。")
else:
    missing = [p for _, p in FEATURE_NOTEBOOKS + [COMPRESSION_NOTEBOOK] if not os.path.exists(os.path.join(PROJECT_ROOT, p))]
    if missing:
        print("エラー: 次のノートブックが見つかりません。")
        for p in missing:
            print("  ", p)
    else:
        print(f"プロジェクトのフォルダ: {PROJECT_ROOT}")
        print("セル2: 関数の定義が完了しました。")

プロジェクトのフォルダ: c:\Users\0uh2j\Desktop\vscodeで\05_換算前フローフロント温度電圧データ抽出
セル2: 関数の定義が完了しました。


In [7]:
# セル3 --- メインフォルダの選択 ---

if MAIN_FOLDER:
    main_folder = MAIN_FOLDER
else:
    print("フォルダ選択ダイアログを開いています...（画面の裏側に隠れている場合があります）")
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    main_folder = filedialog.askdirectory(title="解析したいCSVファイルが入っている【メインフォルダ】を選択してください")
    root.destroy()

if not main_folder:
    print("キャンセルされました。")
elif not os.path.isdir(main_folder):
    print(f"エラー: フォルダが見つかりません: {main_folder}")
else:
    main_folder = os.path.normpath(main_folder)
    print(f"選択されたフォルダ: {main_folder}")
    print("セル3: 準備完了です。セル4を実行してください。")

フォルダ選択ダイアログを開いています...（画面の裏側に隠れている場合があります）
選択されたフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\02_202609_粉砕材混合PP加工処理データ\mixpp_温度データ\mixpp_NR_T\csv\3data
セル3: 準備完了です。セル4を実行してください。


In [8]:
# セル4 --- 一括実行 ---

if not globals().get('PROJECT_ROOT'):
    print("エラー: プロジェクトのフォルダが見つかっていません。セル2の表示を確認してください。")
elif not globals().get('main_folder') or not os.path.isdir(main_folder):
    print("エラー: セル3でメインフォルダを選んでください。")
else:
    run_dir = os.path.abspath(os.path.join(BATCH_DIR, OUTPUT_ROOT, f"{os.path.basename(main_folder)}_{datetime.now():%Y%m%d_%H%M%S}"))
    os.makedirs(run_dir, exist_ok=True)
    summary = []   # (名前, 状態, 詳細, 所要時間[s])
    start_all = time.time()

    with open(os.path.join(run_dir, '実行ログ.txt'), 'w', encoding='utf-8') as log, \
         contextlib.redirect_stdout(Tee(sys.stdout, log)):
        print(f"メインフォルダ: {main_folder}")
        print(f"結果の保存先　: {run_dir}")
        csv_files = []

        for name, nb_path in FEATURE_NOTEBOOKS:
            print(f"\n{'=' * 60}\n■ {name}（{nb_path}）\n{'=' * 60}")
            out_dir = os.path.join(run_dir, name)
            os.makedirs(out_dir, exist_ok=True)
            t0 = time.time()
            try:
                csv_files, summary_path = run_feature_notebook(nb_path, main_folder, out_dir)
                result = pd.read_csv(summary_path)
                blank = result.loc[result.isna().any(axis=1), '元ファイル'].tolist()
                detail = f"{len(csv_files)} ファイル中 {len(result)} 件を出力"
                if blank:
                    detail += f"／空欄のあるファイル: {', '.join(blank)}"
                status = 'OK' if len(result) == len(csv_files) and not blank else '要確認'
                summary.append((name, status, detail, time.time() - t0))
            except Exception:
                traceback.print_exc(file=sys.stdout)
                summary.append((name, 'エラー', traceback.format_exc().strip().splitlines()[-1], time.time() - t0))

        if RUN_COMPRESSION:
            name, nb_path = COMPRESSION_NOTEBOOK
            print(f"\n{'=' * 60}\n■ {name}（{nb_path}）\n{'=' * 60}")
            out_dir = os.path.join(run_dir, name)
            t0 = time.time()
            if not csv_files:
                summary.append((name, 'エラー', '処理する CSV の一覧がありません。', 0.0))
            else:
                try:
                    failed = run_compression(nb_path, main_folder, csv_files, out_dir)
                    done = len(glob.glob(os.path.join(out_dir, '**', '*_smoothed_10k.csv'), recursive=True))
                    detail = f"{len(csv_files)} ファイル中 {done} 件を出力"
                    if failed:
                        detail += f"／失敗したファイル: {', '.join(failed)}"
                    summary.append((name, 'OK' if done == len(csv_files) else '要確認', detail, time.time() - t0))
                except Exception:
                    traceback.print_exc(file=sys.stdout)
                    summary.append((name, 'エラー', traceback.format_exc().strip().splitlines()[-1], time.time() - t0))

        print(f"\n{'=' * 60}\n■ 要約（全体の所要時間 {(time.time() - start_all) / 60:.1f} 分）\n{'=' * 60}")
        for name, status, detail, sec in summary:
            print(f"[{status}] {name}: {detail}（{sec / 60:.1f} 分）")
        print(f"\n結果の保存先: {run_dir}")
        print("セル4: 一括実行が完了しました。")

メインフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\02_202609_粉砕材混合PP加工処理データ\mixpp_温度データ\mixpp_NR_T\csv\3data
結果の保存先　: c:\Users\0uh2j\Desktop\vscodeで\05_換算前フローフロント温度電圧データ抽出\全コード一括実行\実行結果\3data_20261008_084112

■ 最高値版（最高値version/コードフォルダ/電圧特徴量(最高値他)抽出.ipynb）
セル1: ライブラリのインポートが完了しました。
セル2: 設定の読み込みが完了しました。
セル3: 関数の定義が完了しました。
フォルダ選択ダイアログを開いています...（画面の裏側に隠れている場合があります）
選択されたフォルダ: C:\Users\0uh2j\Desktop\実験データ2026\01_本実験データ\02_202609_粉砕材混合PP加工処理データ\mixpp_温度データ\mixpp_NR_T\csv\3data
108個のCSVファイルを検出しました。
セル4: 準備完了です。

データの抽出を開始します（グラフ描画なし・高速）...

--- サブフォルダ: 0.5mm\190℃ ---
抽出中: 001_190℃_280_0.5mm.csv
抽出中: 002_190℃_280_0.5mm.csv
抽出中: 003_190℃_280_0.5mm.csv
抽出中: 004_190℃_320_0.5mm.csv
抽出中: 005_190℃_320_0.5mm.csv
抽出中: 006_190℃_320_0.5mm.csv

--- サブフォルダ: 0.5mm\210℃ ---
抽出中: 001_210℃_280_0.5mm.csv
抽出中: 002_210℃_280_0.5mm.csv
抽出中: 003_210℃_280_0.5mm.csv
抽出中: 004_210℃_320_0.5mm.csv
抽出中: 005_210℃_320_0.5mm.csv
抽出中: 006_210℃_320_0.5mm.csv

--- サブフォルダ: 0.5mm\230℃ ---
抽出中: 001_230℃_280_0.5mm.csv
抽出中: 002_230℃